# 04 — Enforce Constraints with PySpark/SQL (The "Stored Procedure" Approach)

If you're used to Oracle or SQL Server, you've probably enforced data quality through
a combination of **stored procedures**, **triggers**, and **CHECK constraints**. The
database engine was your safety net — no bad data got through without hitting a wall.

On Databricks, most teams write data with PySpark and Spark SQL — so we need to build
that same discipline into our write logic. Think of these patterns as the lakehouse
equivalent of your stored procedure layer:

1. **`NOT EXISTS` filtering** — like a stored proc that checks before inserting
2. **Post-write validation** — like an AFTER INSERT trigger that audits the table
3. **`MERGE` (upsert)** — the single most important pattern, replaces INSERT-or-UPDATE logic
4. **FK validation** — manual referential integrity checks before inserting child rows

These all work. But as we'll see, they have the same problem as stored procedures:
**they only protect you if every writer uses them.**

In [ ]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "pk_unique_demo")

CUSTOMERS_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.customers"
ORDERS_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.orders"
BATCH_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.new_customers_batch"

## Reset customers to clean state

First, we need to undo the damage from notebook 03. We'll overwrite the customers
table with the original 200 rows and reset orders to the original 500.

In [ ]:
# Delete the constraint-violating rows by keeping only the original data
# We know original customer_ids were 1-200 and original order_ids were 1-500
spark.sql(f"DELETE FROM {CUSTOMERS_TABLE} WHERE customer_id > 200 OR customer_id < 1")

# Remove the duplicate rows for original IDs (keep one per customer_id)
spark.sql(f"""
    DELETE FROM {CUSTOMERS_TABLE}
    WHERE email = 'pk_violation@test.com'
""")

# Use MERGE with self to deduplicate: keep the row with the earliest signup_date
spark.sql(f"""
    CREATE OR REPLACE TABLE {CUSTOMERS_TABLE} AS
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY signup_date) AS rn
        FROM {CUSTOMERS_TABLE}
    )
    WHERE rn = 1
""")

# Drop the rn column
spark.sql(f"ALTER TABLE {CUSTOMERS_TABLE} DROP COLUMN rn")

# Deduplicate by email too
spark.sql(f"""
    CREATE OR REPLACE TABLE {CUSTOMERS_TABLE} AS
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY email ORDER BY customer_id) AS rn
        FROM {CUSTOMERS_TABLE}
    )
    WHERE rn = 1
""")

spark.sql(f"ALTER TABLE {CUSTOMERS_TABLE} DROP COLUMN rn")

# Reset orders
spark.sql(f"DELETE FROM {ORDERS_TABLE} WHERE order_id > 500 OR order_id < 1")

cust_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CUSTOMERS_TABLE}").collect()[0].cnt
order_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {ORDERS_TABLE}").collect()[0].cnt
print(f"Clean state: {cust_count} customers, {order_count} orders")

## Re-add constraints (CRAS drops them)

In [ ]:
constraint_stmts = [
    f"ALTER TABLE {CUSTOMERS_TABLE} ADD CONSTRAINT customers_pk PRIMARY KEY (customer_id)",
    f"ALTER TABLE {CUSTOMERS_TABLE} ADD CONSTRAINT customers_email_unique UNIQUE (email)",
    f"ALTER TABLE {ORDERS_TABLE} ADD CONSTRAINT orders_pk PRIMARY KEY (order_id)",
    f"ALTER TABLE {ORDERS_TABLE} ADD CONSTRAINT orders_customer_fk FOREIGN KEY (customer_id) REFERENCES {CUSTOMERS_TABLE}(customer_id)",
]

for stmt in constraint_stmts:
    try:
        spark.sql(stmt)
    except Exception as e:
        # Constraint may already exist if CRAS didn't drop it
        print(f"Note: {e}")

print("Constraints re-applied.")

## Pattern 1: Pre-insert validation with NOT EXISTS

This is the lakehouse equivalent of a stored procedure that does a lookup before inserting.
In Oracle you might write `IF NOT EXISTS (SELECT ...) THEN INSERT ...` inside a PL/SQL block.
Here, we push that logic into the SQL itself — only insert rows that don't collide.

In [ ]:
# Only insert rows where customer_id doesn't already exist
result = spark.sql(f"""
    INSERT INTO {CUSTOMERS_TABLE}
    SELECT b.*
    FROM {BATCH_TABLE} b
    WHERE NOT EXISTS (
        SELECT 1 FROM {CUSTOMERS_TABLE} c
        WHERE c.customer_id = b.customer_id
    )
    AND NOT EXISTS (
        SELECT 1 FROM {CUSTOMERS_TABLE} c
        WHERE c.email = b.email
    )
""")

new_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CUSTOMERS_TABLE}").collect()[0].cnt
print(f"After NOT EXISTS insert: {new_count} customers")
print("Only genuinely new rows were inserted — duplicates filtered out!")

## Pattern 2: Post-write validation function

This is the lakehouse equivalent of an **AFTER INSERT trigger**. Instead of the engine
automatically firing a check, we run it ourselves after the write. If duplicates slipped
through, we catch them and raise an error — just like a trigger would raise `RAISE_APPLICATION_ERROR`
in Oracle or `RAISERROR` in SQL Server.

In [ ]:
def validate_uniqueness(table, columns):
    """Check a table for duplicate values in the given columns. Raises ValueError if found."""
    violations = {}
    for col in columns:
        dupes = spark.sql(f"""
            SELECT {col}, COUNT(*) AS cnt
            FROM {table}
            GROUP BY {col}
            HAVING COUNT(*) > 1
        """)
        dupe_count = dupes.count()
        if dupe_count > 0:
            violations[col] = dupe_count
            print(f"VIOLATION: {dupe_count} duplicate values found in {col}:")
            dupes.show(5, truncate=False)

    if violations:
        raise ValueError(
            f"Uniqueness violations in {table}: "
            + ", ".join(f"{col} ({cnt} duplicates)" for col, cnt in violations.items())
        )
    print(f"✓ No duplicates found in {table} for columns: {columns}")


# Validate our current (clean) state
validate_uniqueness(CUSTOMERS_TABLE, ["customer_id", "email"])

In [ ]:
# Now deliberately insert duplicates and watch the validation catch them
spark.sql(f"""
    INSERT INTO {CUSTOMERS_TABLE}
    SELECT * FROM {BATCH_TABLE}
""")

try:
    validate_uniqueness(CUSTOMERS_TABLE, ["customer_id", "email"])
except ValueError as e:
    print(f"\nCAUGHT: {e}")
    print("\nPost-write validation detected the problem — now we can act on it.")

## Pattern 3: MERGE for upsert semantics

If you've ever written an Oracle `MERGE INTO ... USING ... WHEN MATCHED THEN UPDATE WHEN NOT MATCHED THEN INSERT`,
this is the exact same thing. `MERGE` is the single most important pattern for maintaining
unique keys on Databricks — it handles "insert if new, update if exists" in one atomic operation.

No duplicates possible. No race conditions. This is your go-to.

In [ ]:
# Reset to clean state first
spark.sql(f"""
    CREATE OR REPLACE TABLE {CUSTOMERS_TABLE} AS
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY signup_date) AS rn
        FROM {CUSTOMERS_TABLE}
    )
    WHERE rn = 1
""")
spark.sql(f"ALTER TABLE {CUSTOMERS_TABLE} DROP COLUMN rn")

# Deduplicate by email
spark.sql(f"""
    CREATE OR REPLACE TABLE {CUSTOMERS_TABLE} AS
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY email ORDER BY customer_id) AS rn
        FROM {CUSTOMERS_TABLE}
    )
    WHERE rn = 1
""")
spark.sql(f"ALTER TABLE {CUSTOMERS_TABLE} DROP COLUMN rn")

before = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CUSTOMERS_TABLE}").collect()[0].cnt
print(f"Before MERGE: {before} customers")

In [ ]:
# MERGE: update existing, insert new — no duplicates possible
spark.sql(f"""
    MERGE INTO {CUSTOMERS_TABLE} AS target
    USING {BATCH_TABLE} AS source
    ON target.customer_id = source.customer_id
    WHEN MATCHED THEN UPDATE SET
        target.email = source.email,
        target.first_name = source.first_name,
        target.last_name = source.last_name,
        target.phone = source.phone,
        target.city = source.city,
        target.state = source.state,
        target.signup_date = source.signup_date
    WHEN NOT MATCHED THEN INSERT *
""")

after = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CUSTOMERS_TABLE}").collect()[0].cnt
print(f"After MERGE: {after} customers")

# Verify no duplicates on customer_id
dup_check = spark.sql(f"""
    SELECT customer_id, COUNT(*) AS cnt
    FROM {CUSTOMERS_TABLE}
    GROUP BY customer_id
    HAVING COUNT(*) > 1
""")
print(f"Duplicate customer_ids after MERGE: {dup_check.count()}")
print("MERGE guarantees no PK duplicates by design!")

## Pattern 4: FK validation before inserting orders

In the RDBMS world, the engine checks foreign keys automatically — you never think about it.
Here, we have to do the lookup ourselves. Think of this as the referential integrity check
that used to live inside your stored procedure: "before inserting an order, verify the
customer exists."

In [ ]:
# Create a small batch of new orders — some with valid, some with invalid customer_ids
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW new_orders AS
    SELECT * FROM VALUES
        (10001, 1,    'Laptop',    1, 999.99, DATE '2026-03-01'),
        (10002, 2,    'Keyboard',  2, 49.99,  DATE '2026-03-01'),
        (10003, -999, 'Ghost Pen', 1, 5.99,   DATE '2026-03-01'),
        (10004, -888, 'Ghost Pad', 1, 12.99,  DATE '2026-03-01')
    AS t(order_id, customer_id, product_name, quantity, price, order_date)
""")

In [ ]:
# Show which orders will be accepted vs rejected
print("Orders with VALID customer_id (will be inserted):")
spark.sql(f"""
    SELECT no.*
    FROM new_orders no
    WHERE no.customer_id IN (SELECT customer_id FROM {CUSTOMERS_TABLE})
""").show(truncate=False)

print("Orders with INVALID customer_id (will be rejected):")
spark.sql(f"""
    SELECT no.*
    FROM new_orders no
    WHERE no.customer_id NOT IN (SELECT customer_id FROM {CUSTOMERS_TABLE})
""").show(truncate=False)

In [ ]:
# Insert only the valid ones
spark.sql(f"""
    INSERT INTO {ORDERS_TABLE}
    SELECT no.*
    FROM new_orders no
    WHERE no.customer_id IN (SELECT customer_id FROM {CUSTOMERS_TABLE})
""")

# Verify no orphans
orphans = spark.sql(f"""
    SELECT o.order_id, o.customer_id
    FROM {ORDERS_TABLE} o
    LEFT JOIN {CUSTOMERS_TABLE} c ON o.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
""")
print(f"Orphaned orders: {orphans.count()}")
print("FK validation prevented invalid references!")

## Takeaway: This works, but it has the same flaw as stored procedures

Every pattern above is effective — **if every writer uses it.** But think about your
Oracle or SQL Server environment honestly:

- How many times did someone bypass the stored procedure and do a raw `INSERT`?
- How many ad-hoc scripts skipped the trigger by using `ALTER TABLE ... DISABLE TRIGGER`?
- How many ETL jobs did a bulk load that bypassed constraints entirely?

The same problem exists here, except it's worse: there's no engine-level safety net at all.
If a new team member writes a Spark job that does a plain `INSERT INTO` instead of using
`MERGE`, duplicates go right in and nobody gets an error.

### What we really need

We need something that **sits at the pipeline level** — a single gatekeeper that enforces
data quality rules regardless of who or what is writing. Something that:

- Is declared once, not scattered across every writer's code
- Can't be bypassed by a careless `INSERT`
- Gives you clear observability into what failed and why
- Can either warn, drop bad rows, or halt the pipeline entirely

That's exactly what **DLT expectations** are. Think of them as **engine-level constraints
that live in your pipeline definition instead of in the storage layer.** Next notebook.